In [ ]:
pip install duckdb pandas scikit-learn numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 38.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 45.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import duckdb
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# ==========================================
# 1. DATA INGESTION & FEATURE ENGINEERING
# ==========================================
print("Connecting to DuckDB and processing data...")

# We use DuckDB to join the parquet and csv files efficiently
query = """
WITH call_aggs AS (
    SELECT 
        unique_customer_identifier,
        SUM(talk_time_seconds) AS total_talk_time,
        SUM(hold_time_seconds) AS total_hold_time,
        COUNT(*) AS total_calls,
        SUM(CASE WHEN call_type = 'Loyalty' THEN 1 ELSE 0 END) AS loyalty_calls
    FROM read_csv_auto('calls.csv', sample_size=-1, ignore_errors=true)
    GROUP BY 1
),
usage_aggs AS (
    SELECT 
        unique_customer_identifier,
        -- CAST applied here to fix the VARCHAR error
        AVG(CAST(usage_download_mbs AS FLOAT)) AS avg_download_mbs,
        AVG(CAST(usage_upload_mbs AS FLOAT)) AS avg_upload_mbs
    FROM read_parquet('usage.parquet')
    GROUP BY 1
),
cease_flags AS (
    SELECT DISTINCT unique_customer_identifier, 1 AS is_churn
    FROM read_csv_auto('cease.csv', sample_size=-1, ignore_errors=true)
)

SELECT 
    c.*,
    COALESCE(ca.total_talk_time, 0) AS total_talk_time,
    COALESCE(ca.total_hold_time, 0) AS total_hold_time,
    COALESCE(ca.total_calls, 0) AS total_calls,
    COALESCE(ca.loyalty_calls, 0) AS loyalty_calls,
    COALESCE(ua.avg_download_mbs, 0) AS avg_download_mbs,
    COALESCE(ua.avg_upload_mbs, 0) AS avg_upload_mbs,
    -- Target Variable
    COALESCE(cf.is_churn, 0) AS is_churn,
    -- CAST applied here as well just to be safe with speed metrics
    (CAST(c.speed AS FLOAT) - CAST(c.line_speed AS FLOAT)) AS speed_deficit
FROM read_parquet('customer_info.parquet') c
LEFT JOIN call_aggs ca ON c.unique_customer_identifier = ca.unique_customer_identifier
LEFT JOIN usage_aggs ua ON c.unique_customer_identifier = ua.unique_customer_identifier
LEFT JOIN cease_flags cf ON c.unique_customer_identifier = cf.unique_customer_identifier
"""

# Execute query and load into a Pandas DataFrame
df = duckdb.query(query).df()
print(f"Data processed. Total records: {len(df)}")

# ==========================================
# 2. MODEL TRAINING (RANDOM FOREST)
# ==========================================
print("Training Churn Prediction Model...")

# Select features for the model
features = [
    'contract_dd_cancels', 'dd_cancel_60_day', 'ooc_days', 
    'tenure_days', 'speed_deficit', 'total_talk_time', 
    'total_hold_time', 'loyalty_calls', 'avg_download_mbs'
]

# Fill missing values for modelling
X = df[features].fillna(0)
y = df['is_churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest (interpretable and robust)
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Predict risk probabilities (Probability of Churn)
df['churn_risk_score'] = model.predict_proba(X)[:, 1]

# ==========================================
# 3. EXTRACTING OUTPUTS FOR LOVABLE
# ==========================================

# --- Output 1: feature_importance.json ---
print("Generating Feature Importance...")
importances = model.feature_importances_
feature_importance_dict = [
    {"feature": f, "importance": round(float(imp), 4)} 
    for f, imp in zip(features, importances)
]
# Sort by highest importance
feature_importance_dict = sorted(feature_importance_dict, key=lambda x: x['importance'], reverse=True)[:10]

with open('feature_importance.json', 'w') as f:
    json.dump(feature_importance_dict, f, indent=4)


# --- Output 2: segment_risk_summary.csv ---
print("Generating Segment Risk Summary...")
# Define Risk Tiers based on percentiles or fixed thresholds
conditions = [
    (df['churn_risk_score'] >= 0.7),
    (df['churn_risk_score'] >= 0.3) & (df['churn_risk_score'] < 0.7),
    (df['churn_risk_score'] < 0.3)
]
choices = ['High Risk', 'Medium Risk', 'Low Risk']
df['Risk_Tier'] = np.select(conditions, choices, default='Low Risk')

# Group to create the summary
segment_summary = df.groupby('Risk_Tier').agg(
    Customer_Count=('unique_customer_identifier', 'count'),
    Average_Tenure_Days=('tenure_days', 'mean'),
    Average_Risk_Score=('churn_risk_score', 'mean'),
    # Get the most common package in each tier
    Dominant_Package=('crm_package_name', lambda x: x.mode()[0] if not x.empty else 'Unknown')
).reset_index()

# Sort for logical presentation
segment_summary['Sort_Order'] = segment_summary['Risk_Tier'].map({'High Risk': 1, 'Medium Risk': 2, 'Low Risk': 3})
segment_summary = segment_summary.sort_values('Sort_Order').drop(columns=['Sort_Order'])
segment_summary.to_csv('segment_risk_summary.csv', index=False)


# --- Output 3: nba_roi_params.json ---
print("Generating ROI Baseline Parameters...")
# Deriving some baseline commercial metrics for the Finance sliders
total_customers = len(df)
high_risk_customers = len(df[df['Risk_Tier'] == 'High Risk'])

# Mocking ARPU (Average Revenue Per User) based on standard UK broadband rates (£35/month * 12)
average_annual_arpu = 420.0 

roi_params = {
    "total_customer_base": total_customers,
    "high_risk_volume": high_risk_customers,
    "average_annual_arpu_gbp": average_annual_arpu,
    "baseline_retention_conversion_rate": 0.15, # 15% default assumed success rate for NBA
    "revenue_at_risk_gbp": high_risk_customers * average_annual_arpu
}

with open('nba_roi_params.json', 'w') as f:
    json.dump(roi_params, f, indent=4)

print("✅ Success! All Lovable assets generated successfully.")

Connecting to DuckDB and processing data...
Data processed. Total records: 3545538
Training Churn Prediction Model...
Generating Feature Importance...
Generating Segment Risk Summary...
Generating ROI Baseline Parameters...
✅ Success! All Lovable assets generated successfully.


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# ==========================================
# 4. MODEL EVALUATION & EXPORTING STATS
# ==========================================
print("Evaluating model performance on test holdout...")

# Generate predictions on the test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate core metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Calculate Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Package into a clean JSON for Lovable
training_stats = {
    "model_type": "Random Forest Classifier",
    "hyperparameters": {
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42
    },
    "performance_metrics": {
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "roc_auc": round(roc_auc, 4)
    },
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    },
    "dataset_split": {
        "train_size": len(X_train),
        "test_size": len(X_test)
    }
}

with open('model_training_stats.json', 'w') as f:
    json.dump(training_stats, f, indent=4)

print("Training statistics exported to model_training_stats.json")

Evaluating model performance on test holdout...
Training statistics exported to model_training_stats.json
